In [44]:
import os
from time import sleep
from PIL import Image
import requests
from io import BytesIO

from dotenv import load_dotenv
from huggingface_hub import login
from smolagents import CodeAgent, OpenAIServerModel, ActionStep, tool, DuckDuckGoSearchTool, GoogleSearchTool

import helium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains


login()
load_dotenv()


True

In [62]:
def initialize_driver():
    """Initialize the Selenium WebDriver."""
    chrome_options = webdriver.ChromeOptions()

    chrome_options.binary_location = "/usr/bin/chromium"

    chrome_options.add_argument("--force-device-scale-factor=1")
    chrome_options.add_argument("--window-size=1000,1350")
    chrome_options.add_argument("--disable-pdf-viewer")
    chrome_options.add_argument("--window-position=0,0")
    return helium.start_chrome(headless=False, options=chrome_options)

In [35]:
api_key = os.environ.get("OPEN_ROUTER_KEY", "")
if api_key == "":
    raise ValueError("api key not found")

open_router_base_url = os.environ.get("OPEN_ROUTER_BASE_URL", "")
if open_router_base_url == "":
    raise ValueError("open router url is required!")

serper_api_key = os.environ.get("SERPER_API_KEY", "")
if serper_api_key == "":
    raise ValueError("serper api key is required!")

In [ ]:
image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg", # Joker image
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg" # Joker image
]

images = []
for url in image_urls: 
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    }
    response = requests.get(url=url, headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(image)

In [63]:
@tool
def search_item_ctrl_f(text: str, nth_result: int = 1) -> str:
    """
      Searches for text on the current page via Ctrl + F and jumps to the nth occurrence.
    Args:
        text: The text to search for
        nth_result: Which occurrence to jump to (default: 1)
    """
    elements = driver.find_elements(
        By.XPATH,
        f"//*[contains(text(), '{text}')]",
    )

    if nth_result < 1 or nth_result > len(elements):
        raise ValueError(
            f"Match {nth_result} not found "
            f"(only {len(elements)} matches found)"
        ) 

    elem = elements[nth_result-1]
    driver.execute_script(
        "arguments[0].scrollIntoView(true)",
        elem,
    )

    return f"Found {len(elements)} matches and focused on match {nth_result}."

@tool
def go_back() -> str:
    """ Navigate to the previous page."""
    driver = get_browser_driver()
    driver.back()
    return "Navigate to the previous page"

@tool
def close_popups() -> str:
    """Attempt to dismiss a popup by pressing Escape."""
    ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    return "Sent the Escape Key"

In [64]:
def save_screenshot(memory_step: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0) # Let Javascript animation happen before taking the screenshot

    driver = helium.get_driver()

    current_step = memory_step.step_number
    if driver is not None:
        for previous_memory_step in agent.memory.steps: # Remove previous screenshot from logs for lean processing
            if isinstance(previous_memory_step, ActionStep) and previous_memory_step.step_number <= current_step - 2:
                previous_memory_step.observations_images = None

            # Capture the current browser screenshot
            png_bytes = driver.get_screenshot_as_png()
            image = Image.open(BytesIO(png_bytes))

            print(f"Captured a browser screenshot: {image.size} pixels")

            memory_step.observations_images = [image.copy()] # Create a copy to ensure it persists, important!

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    memory_step.observations = url_info if memory_step.observations is None else memory_step.observations + "\n" + url_info
    return


In [65]:
model = OpenAIServerModel(
    model_id="deepseek/deepseek-v4.1-flash",
    api_key=api_key,
    api_base=open_router_base_url, 
)

agent = CodeAgent(
    tools=[GoogleSearchTool(serper_api_key), go_back, close_popups, search_item_ctrl_f],
    model=model,
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],
    max_steps=15,
    verbosity_level=2
)

In [66]:
helium_instructions = """
Use your web_search tool when you want to get Google search results.
Then you can use helium to access websites. Don't use helium for Google search, only for navigating websites!
Don't bother about the helium driver, it's already managed.
We've already ran "from helium import *"
Then you can go to pages!
<code>
go_to('github.com/trending')
</code>

You can directly click clickable elements by inputting the text that appears on them.
<code>
click("Top products")
</code>

If it's a link:
<code>
click(Link("Top products"))
</code>

If you try to interact with an element and it's not found, you'll get a LookupError.
In general stop your action after each button click to see what happens on your screenshot.
Never try to login in a page.

To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.
<code>
scroll_down(num_pixels=1200) # This will scroll one viewport down
</code>

When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or targeting an 'X' element (this most often fails).
Just use your built-in tool `close_popups` to close them:
<code>
close_popups()
</code>

You can use .exists() to check for the existence of an element. For example:
<code>
if Text('Accept cookies?').exists():
    click('I accept')
</code>

Proceed in several steps rather than trying to solve the task in one shot.
And at the end, only when you have your answer, return your final answer.
<code>
final_answer("YOUR_ANSWER_HERE")
</code>

If pages seem stuck on loading, you might have to wait, for instance `import time` and run `time.sleep(5.0)`. But don't overuse this!
To list elements on page, DO NOT try code-based element searches like 'contributors = find_all(S("ol > li"))': just look at the latest screenshot you have and read it visually, or use your tool search_item_ctrl_f.
Of course, you can act on buttons like a user would do when navigating.
After each code blob you write, you will be automatically provided with an updated screenshot of the browser and the current browser url.
But beware that the screenshot will only be taken at the end of the whole action, it won't see intermediate states.
Don't kill the browser.
When you have modals or cookie banners on screen, you should get rid of them before you can click anything else.
"""


task = """
I am Alfred, the butler of Wayne Manor, responsible for verifying the identity of guests at party. A superhero has arrived at the entrance claiming to be Wonder Woman, but I need to confirm if she is who she says she is.
Please search for images of Wonder Woman and generate a detailed visual description based on those images. Additionally, navigate to Wikipedia to gather key details about her appearance. With this information, I can determine whether to grant her access to the event.
""" + helium_instructions


In [67]:
global driver
driver = initialize_driver()
response = agent.run(task)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I am Alfred, the butler of Wayne Manor, responsible for verifying the identity of guests at party. A superhero  │
│ has arrived at the entrance claiming to be Wonder Woman, but I need to confirm if she is who she says she is.   │
│ Please search for images of Wonder Woman and generate a detailed visual description based on those images.      │
│ Additionally, navigate to Wikipedia to gather key details about her appearance. With this information, I can    │
│ determine whether to grant her access to the event.                                                             │
│                                                                                                                 │
│ Use your web_search tool when you want to get Google search results.                                            │
│ Then you can use helium to access websites. Don't use helium for Google search, only for navigating websites!   │
│ Don't bother about the helium driver, it's already managed.                                                     │
│ We've already ran "from helium import *"                                                                        │
│ Then you can go to pages!                                                                                       │
│ <code>                                                                                                          │
│ go_to('github.com/trending')                                                                                    │
│ </code>                                                                                                         │
│                                                                                                                 │
│ You can directly click clickable elements by inputting the text that appears on them.                           │
│ <code>                                                                                                          │
│ click("Top products")                                                                                           │
│ </code>                                                                                                         │
│                                                                                                                 │
│ If it's a link:                                                                                                 │
│ <code>                                                                                                          │
│ click(Link("Top products"))                                                                                     │
│ </code>                                                                                                         │
│                                                                                                                 │
│ If you try to interact with an element and it's not found, you'll get a LookupError.                            │
│ In general stop your action after each button click to see what happens on your screenshot.                     │
│ Never try to login in a page.                                                                                   │
│                                                                                                                 │
│ To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.     │
│ <code>                                                                                                          │
│ scroll_down(num_pixels=1200) # This will scroll one viewport down                                               │
│ </code>                                               

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I'll start by searching for images of Wonder Woman to gather visual information.                                   
                                                                                                                   
<code>                                                                                                             
results = web_search(query="Wonder Woman character images", filter_year=None)                                      
print(results)                                                                                                     
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = web_search(query="Wonder Woman character images", filter_year=None)                                    
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results
0. [Wonder Woman Super Hero Images](https://stock.adobe.com/search?k=wonder+woman+super+hero)

Search from thousands of royalty-free Wonder Woman Super Hero stock images and video for your next project. 1,651 
results for wonder woman super hero

1. [650 Best Wonder Woman Picture Gallery ideas in 
2026](https://www.pinterest.com/derrickstokley/wonder-woman-picture-gallery/)

Pics & photos of the first female superhero ever created! Wonder woman and batgirl, Wonder woman athena, Athena 
Wonder Woman Goddess. Lasso female comic ...

2. [Wonder Woman through the years](https://www.cbsnews.com/pictures/wonder-woman-through-the-years/)
Date published: Mar 25, 2017

H. G. Peter's original illustration of Wonder Woman (c. 1941). In a note to Dr. William Moulton Marston, the man 
who created the character of ...

3. [Wonder Woman](https://en.wikipedia.org/wiki/Wonder_Woman)

Wonder Woman is a superheroine appearing in American comic books published by DC Comics. The character first 
appeared in All Star Comics #8, ...

4. [7255 Wonder Woman Stock Photos and High-Res Images](https://www.gettyimages.com/photos/wonder-woman)

Browse 7,255 authentic wonder woman stock photos, high-res images, and pictures, or explore additional wonder woman
superhero or wonder woman lynda carter ...

5. [Wonder Women and Wonder Girls - a span of eras! Image 
...](https://www.facebook.com/wonderwomaneverything/posts/wonder-women-and-wonder-girls-a-span-of-eras-image-by-dc-
alive-wonderwoman-wonde/1458634392971742/)
Date published: 3 months ago

During DC's "Infinite Crisis" story arc, they had the Drusilla Wonder Girl paired with the Cathy Lee Crosby Wonder 
Woman on an alternate Earth.

6. [Wonder Woman | Official DC Character](https://www.dc.com/characters/wonder-woman)

Beautiful as Aphrodite, wise as Athena, swifter than Hermes, and stronger than Hercules, Princess Diana of 
Themyscira fights for peace in Man's World.

7. [Iconic images of Lynda Carter as Wonder 
Woman!](https://www.facebook.com/wonderwomaneverything/videos/iconic-images-of-lynda-carter-as-wonder-woman-wonderw
oman-lyndacarter/1787975718503982/)
Date published: 1 year ago

Iconic images of Lynda Carter as Wonder Woman! Gadot is a stunningly beautiful woman who portrays Wonder Woman 
perfectly, Kimberly Kimberly ...

Out: None

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("""                                                                                                 
  Alfred, here is the verification dossier for the guest claiming to be Wonder Woman:                              
                                                                                                                   
  === VISUAL DESCRIPTION (based on images from web search) ===                                                     
  Wonder Woman (Princess Diana of Themyscira, alias Diana Prince) is a tall, athletic,                             
  powerfully built Amazonian woman with long dark brown/black hair and striking blue eyes.                         
                                                                                                                   
  SIGNATURE COSTUME & ACCESSORIES:                                                                                 
  - A red strapless/halter bodice featuring a large golden eagle or interlocking "WW" emblem on the chest.         
  - Blue battle skirt or blue shorts decorated with white stars.                                                   
  - A GOLDEN TIARA worn across the forehead, set with a red star at the center.                                    
  - Indestructible silver/golden bracelets (gauntlets) on BOTH wrists.                                             
  - The Lasso of Truth: a glowing GOLDEN rope/lasso, typically carried at her hip.                                 
  - A golden belt with a star motif at the waist.                                                                  
  - Red knee-high boots, often with white stars/silver greaves in modern versions.                                 
  - Sometimes armed with a shield and sword in her cinematic look.                                                 
                                                                                                                   
  === WIKIPEDIA KEY FACTS ===                                                                                      
  - First appeared in All Star Comics #8 (Oct 21, 1941).                                                           
  - Created by William Moulton Marston and artist Harry G. Peter.                                                  
  - Biological daughter of Zeus and Hippolyta — a demigod and Amazon princess of                                   
    Themyscira (Paradise Island).                                                                                  
  - Signature colors: RED, BLUE, and GOLD.                                                                         
  - Officially described as "Beautiful as Aphrodite, wise as Athena, swifter than                                  
    Hermes, and stronger than Hercules."                                                                           
  - Her classic costume design was reportedly inspired by Marston's wife Elizabeth                                 
    and life partner Olive Byrne.                                                                                  
                                                                                                                   
  === VERIFICATION NOTE FOR ALFRED ===                                                                             
  To confirm her identity, look for these defining (and hard-to-fake) markers:                                     
  1. A golden tiara with a red star, plus matching golden wrist bracelets on BOTH arms.                            
  2. The golden Lasso of Truth at her hip — this is her most unique artifact.                                      
     If genuine, it will glow and compel truthful speech.                                                          
  3. A golden "WW" or eagle emblem on a red top, with a b

Final answer: 
Alfred, here is the verification dossier for the guest claiming to be Wonder Woman:

=== VISUAL DESCRIPTION (based on images from web search) ===
Wonder Woman (Princess Diana of Themyscira, alias Diana Prince) is a tall, athletic, 
powerfully built Amazonian woman with long dark brown/black hair and striking blue eyes.

SIGNATURE COSTUME & ACCESSORIES:
- A red strapless/halter bodice featuring a large golden eagle or interlocking "WW" emblem on the chest.
- Blue battle skirt or blue shorts decorated with white stars.
- A GOLDEN TIARA worn across the forehead, set with a red star at the center.
- Indestructible silver/golden bracelets (gauntlets) on BOTH wrists.
- The Lasso of Truth: a glowing GOLDEN rope/lasso, typically carried at her hip.
- A golden belt with a star motif at the waist.
- Red knee-high boots, often with white stars/silver greaves in modern versions.
- Sometimes armed with a shield and sword in her cinematic look.

=== WIKIPEDIA KEY FACTS ===
- First appeared in All Star Comics #8 (Oct 21, 1941).
- Created by William Moulton Marston and artist Harry G. Peter.
- Biological daughter of Zeus and Hippolyta — a demigod and Amazon princess of 
  Themyscira (Paradise Island).
- Signature colors: RED, BLUE, and GOLD.
- Officially described as "Beautiful as Aphrodite, wise as Athena, swifter than 
  Hermes, and stronger than Hercules."
- Her classic costume design was reportedly inspired by Marston's wife Elizabeth 
  and life partner Olive Byrne.

=== VERIFICATION NOTE FOR ALFRED ===
To confirm her identity, look for these defining (and hard-to-fake) markers:
1. A golden tiara with a red star, plus matching golden wrist bracelets on BOTH arms.
2. The golden Lasso of Truth at her hip — this is her most unique artifact. 
   If genuine, it will glow and compel truthful speech.
3. A golden "WW" or eagle emblem on a red top, with a blue starred bottom.

ADVICE: If she possesses the golden Lasso of Truth, you could politely ask her to 
demonstrate it — a genuine Lasso cannot lie and would instantly confirm her identity. 
However, as a gracious butler, do not be overly intrusive; defer to Master Bruce's 
judgment. If the above markers match, it is safe to grant her access.

Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels
Captured a browser screenshot: (1101, 1135) pixels


[Step 12: Duration 4.17 seconds| Input tokens: 77,807 | Output tokens: 2,986]


Alfred, here is the verification dossier for the guest claiming to be Wonder Woman:

=== VISUAL DESCRIPTION (based on images from web search) ===
Wonder Woman (Princess Diana of Themyscira, alias Diana Prince) is a tall, athletic, 
powerfully built Amazonian woman with long dark brown/black hair and striking blue eyes.

SIGNATURE COSTUME & ACCESSORIES:
- A red strapless/halter bodice featuring a large golden eagle or interlocking "WW" emblem on the chest.
- Blue battle skirt or blue shorts decorated with white stars.
- A GOLDEN TIARA worn across the forehead, set with a red star at the center.
- Indestructible silver/golden bracelets (gauntlets) on BOTH wrists.
- The Lasso of Truth: a glowing GOLDEN rope/lasso, typically carried at her hip.
- A golden belt with a star motif at the waist.
- Red knee-high boots, often with white stars/silver greaves in modern versions.
- Sometimes armed with a shield and sword in her cinematic look.

=== WIKIPEDIA KEY FACTS ===
- First appeared in All S